## Инициализация

Обязательная ячейка для подключения к API Mediascope, загрузки необходимых библиотек и настройки отображения результатов.
Выполняется один раз после открытия или перезапуска ноутбука.

In [ ]:
%reload_ext autoreload
%autoreload 2

import os
import re
import json
import time
import pandas as pd

from datetime import datetime, timedelta, date
from IPython.display import JSON

from mediascope_api.core import net as mscore
from mediascope_api.crossweb import tasks as cwt
from mediascope_api.crossweb import catalogs as cwc

# Отображаем все колонки в таблицах
pd.set_option('display.max_columns', None)

# При необходимости можно увеличить количество отображаемых строк
# pd.set_option('display.max_rows', 200)

# Внимание: отображение всех строк может потребовать много ресурсов
# pd.set_option('display.max_rows', None)

# Создаём объекты для подключения к API и работы с Cross Web
mnet = mscore.MediascopeApiNetwork()
mtask = cwt.CrossWebTask()
cats = cwc.CrossWebCats()

### Параметры отчёта

В ячейке задаются типы использования, география, аудитория, список анализируемых ресурсов и рассчитываемые показатели.

In [ ]:
# Типы использования ресурса
usetype_filter = [1, 2, 3]

# География и социально-демографические характеристики
# None означает, что дополнительный фильтр не применяется
geo_filter = None
demo_filter = None

# ID ресурсов, для которых рассчитывается накопленный охват
mart_filter = 'crossMediaResourceId IN (1094, 3768)'

# Разрез результата
slices = ['crossMediaResourceId']
#slices = ['crossMediaProductId']

# Рассчитываемый показатель
statistics = ['reach']

### Период и шаг расчёта

Укажите начало и окончание периода, единицу и размер шага.

- **dateFrom** — дата начала расчёта.
- **dateTo** — дата окончания расчёта включительно.
- **step_unit** — единица шага: `days` — дни, `weeks` — недели, `months` — месяцы.
- **n** — размер шага, целое число не меньше 1.

Охват рассчитывается накопительно: начало периода остаётся неизменным, а окончание сдвигается на выбранный шаг.

Неделя отсчитывается как 7 дней от даты начала. Для расчёта по полным календарным месяцам задайте начало первым числом месяца.

Последний расчёт выполняется на дату `dateTo`, даже если оставшийся период короче выбранного шага.

In [ ]:
# Начало периода
dateFrom = '2025-03-01'

# Окончание периода включительно
dateTo = '2025-03-11'

# Единица шага: 'days', 'weeks' или 'months'
step_unit = 'days'

# Размер шага: например, 1 месяц, 2 недели или 3 дня
n = 1

### Запуск расчёта

Ячейка проверяет параметры и формирует накопительные периоды с выбранным шагом: в днях, неделях или месяцах.

Для каждого периода отправляется отдельное задание в API. Начало периода остаётся фиксированным, последний период заканчивается на дату `dateTo`.

Дождитесь сообщения о завершении расчёта перед получением результатов.

In [ ]:
# Преобразуем даты
start_date = datetime.strptime(dateFrom, '%Y-%m-%d')
end_date = datetime.strptime(dateTo, '%Y-%m-%d')

# Проверяем параметры
if end_date < start_date:
    raise ValueError('Дата окончания не может быть раньше даты начала.')

if type(n) is not int or n < 1:
    raise ValueError('Размер шага n должен быть целым числом не меньше 1.')

if step_unit not in ('days', 'weeks', 'months'):
    raise ValueError("Единица шага: 'days', 'weeks' или 'months'.")

# Формируем даты окончания накопительных периодов
calculation_dates = []
step_number = 1

while True:
    # Отсчитываем каждый шаг от исходной даты начала
    if step_unit == 'days':
        next_boundary = start_date + timedelta(days=n * step_number)

    elif step_unit == 'weeks':
        next_boundary = start_date + timedelta(weeks=n * step_number)

    else:
        next_boundary = (
            pd.Timestamp(start_date)
            + pd.DateOffset(months=n * step_number)
        ).to_pydatetime()

    # Дата окончания включается в период
    current_end_date = next_boundary - timedelta(days=1)

    # Ограничиваем последний период заданной датой окончания
    if current_end_date >= end_date:
        calculation_dates.append(end_date)
        break

    calculation_dates.append(current_end_date)
    step_number += 1

# Создаём новый список задач
tasks = []

print(f'Отправка задач. Количество периодов: {len(calculation_dates)}')

for current_end_date in calculation_dates:
    current_dateTo = current_end_date.strftime('%Y-%m-%d')
    date_filter = [(dateFrom, current_dateTo)]
    project_name = f'{dateFrom} — {current_dateTo}'

    print(f'Период: {project_name}')

    # Формируем задание для API Cross Web
    task_json = mtask.build_task(
        'media',
        project_name,
        date_filter,
        usetype_filter,
        geo_filter,
        demo_filter,
        mart_filter,
        slices,
        statistics
    )

    # Отправляем задание и сохраняем его в списке
    task_info = {
        'project_name': project_name,
        'task': mtask.send_task(task_json)
    }

    tasks.append(task_info)

    # Пауза между отправками заданий
    time.sleep(2)

print('Все задачи отправлены. Ожидаем завершения расчёта.')

# Ожидаем выполнения заданий
mtask.wait_task(tasks)

print('Расчёт завершён. Можно получать результаты.')

### Получение результатов

Ячейка загружает результаты всех выполненных задач из API и объединяет их в одну таблицу.
Выполняется только после сообщения о завершении расчёта в предыдущей ячейке.

In [ ]:
results = []

for task_info in tasks:
    task_id = task_info['task']
    project_name = task_info['project_name']

    result_table = mtask.result2table(
        mtask.get_result(task_id),
        project_name=project_name
    )

    results.append(result_table)

# Объединяем результаты всех периодов
df_result = pd.concat(results, ignore_index=True)

df_result

### Выгрузка результата

Ячейка сохраняет полученные результаты в файл Excel на лист Report.

Название файла включает название отчёта, даты начала и окончания периода, размер и единицу шага.

Чтобы изменить название отчёта, отредактируйте параметр `report_title` в коде ниже.

Выполняйте ячейку после получения результатов.

In [ ]:
# Название отчёта — этот текст можно менять
report_title = 'Накопленный охват'

# Обозначения единиц шага для названия файла
step_labels = {
    'days': 'дн.',
    'weeks': 'нед.',
    'months': 'мес.'
}

# Автоматически добавляем период и шаг расчёта
report_name = (
    f'{report_title} '
    f'{dateFrom}_{dateTo} '
    f'шаг {n} {step_labels[step_unit]}'
)

excel_filename = mtask.get_excel_filename(report_name)

# Сохраняем результаты в Excel
with pd.ExcelWriter(excel_filename) as writer:
    df_result.to_excel(writer, sheet_name='Report', index=False)

print(f'Отчёт сохранён: {excel_filename}')